# Notebook 02 — Feature Engineering
Reads `data/processed/unified_clean.csv`, applies the full feature pipeline, writes `data/processed/featured.csv`.

In [1]:
import sys, os
# Ensure project root is on path
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd
from src.features import run_full_pipeline, FEATURE_COLS

CLEAN_CSV   = '../data/processed/unified_clean.csv'
FEATURED_CSV = '../data/processed/featured.csv'

if not os.path.exists(CLEAN_CSV):
    raise FileNotFoundError(f'unified_clean.csv not found at {CLEAN_CSV}. Run notebook 01 first.')

df = pd.read_csv(CLEAN_CSV, parse_dates=['date'])
print(f'Loaded {len(df):,} rows from unified_clean.csv')
print('Columns:', df.columns.tolist())

Loaded 18,260 rows from unified_clean.csv
Columns: ['city', 'date', 'aqi', 'pm25', 'pm10', 'no2', 'so2', 'o3', 'co', 'data_quality']


In [2]:
# Run full feature engineering pipeline
featured = run_full_pipeline(df)
print(f'Featured DataFrame: {len(featured):,} rows, {len(featured.columns)} columns')
print('Feature columns present:', [c for c in FEATURE_COLS if c in featured.columns])
print('Missing feature cols:', [c for c in FEATURE_COLS if c not in featured.columns])

Featured DataFrame: 18,240 rows, 27 columns
Feature columns present: ['aqi_yesterday', 'aqi_7day_avg', 'aqi_3day_avg', 'aqi_momentum', 'aqi_3day_change', 'pm25_lag1', 'pm25_lag1_sq', 'pm10_lag1', 'so2_lag1', 'no2_lag1', 'month', 'is_winter', 'is_monsoon', 'is_industrial_peak', 'winter_industrial']
Missing feature cols: []


In [3]:
# Validate no NaN in feature columns
nan_counts = featured[FEATURE_COLS].isna().sum()
print('NaN counts per feature column:')
print(nan_counts[nan_counts > 0] if nan_counts.any() else 'None — all clean')

NaN counts per feature column:
None — all clean


In [4]:
# Write featured.csv
os.makedirs(os.path.dirname(FEATURED_CSV), exist_ok=True)
featured.to_csv(FEATURED_CSV, index=False)
print(f'Written {len(featured):,} rows to {FEATURED_CSV}')
print('Schema:', featured.dtypes.to_dict())

Written 18,240 rows to ../data/processed/featured.csv
Schema: {'city': dtype('O'), 'date': dtype('<M8[ns]'), 'aqi': dtype('float64'), 'pm25': dtype('float64'), 'pm10': dtype('float64'), 'no2': dtype('float64'), 'so2': dtype('float64'), 'o3': dtype('float64'), 'co': dtype('float64'), 'data_quality': dtype('O'), 'aqi_yesterday': dtype('float64'), 'pm25_lag1': dtype('float64'), 'pm10_lag1': dtype('float64'), 'so2_lag1': dtype('float64'), 'no2_lag1': dtype('float64'), 'pm25_lag1_sq': dtype('float64'), 'aqi_7day_avg': dtype('float64'), 'aqi_30day_avg': dtype('float64'), 'aqi_3day_avg': dtype('float64'), 'aqi_momentum': dtype('float64'), 'aqi_3day_change': dtype('float64'), 'aqi_target': dtype('float64'), 'month': dtype('int64'), 'is_winter': dtype('int64'), 'is_monsoon': dtype('int64'), 'is_industrial_peak': dtype('int64'), 'winter_industrial': dtype('int64')}
